# 01 Project: Conference Schedule
Codecademy: Intro to SQL, lesson 1 project "Create a Table".

A tech conference needs a table to track sessions, speakers and rooms as the schedule changes.
Code below is my own solution. Mistakes are flagged as learning points.

In [8]:
# turns on the %%sql cell magic
%load_ext sql
# opens data\conference.db, creates the file if it doesn't exist
%sql sqlite:///../data/conference.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## Task 1. Create the table

Task: create `sessions` with `id`, `title`, `speaker`, `session_date`.
(Codecademy says "three columns" but lists four.)

In [9]:
%%sql
-- DROP TABLE IF EXISTS deletes the table if it's there, so the notebook can be re-run
DROP TABLE IF EXISTS sessions;

CREATE TABLE sessions (
  id INTEGER,
  title TEXT,
  speaker TEXT,
  session_date DATE
);

Running query in 'sqlite:///../data/conference.db'

++
||
++
++

## Tasks 2-3. First session + check

Task: add Maya Chen's "Intro to Data Visualization" on 2026-05-01, then view all sessions.

**Mistake: double quotes for text.** I originally wrote `"Intro to Data Visualization"`.

| Quotes | Standard SQL meaning |
|---|---|
| `'single'` | text value |
| `"double"` | a column or table **name** |

SQLite lets `"text"` slide (it falls back to text if no column has that name), so Codecademy accepted it.
Other databases (PostgreSQL, SQL Server) throw an error. Fixed to single quotes in all cells below.

In [10]:
%%sql
INSERT INTO sessions (id, title, speaker, session_date)
VALUES (1, 'Intro to Data Visualization', 'Maya Chen', '2026-05-01');

SELECT * FROM sessions;

Running query in 'sqlite:///../data/conference.db'

1 rows affected.

id,title,speaker,session_date
1,Intro to Data Visualization,Maya Chen,2026-05-01


## Task 4. Remaining sessions

Task: add Jordan Patal (2026-05-02) and Elena Rivera (2026-05-03). Several rows in one `INSERT`.

In [11]:
%%sql
INSERT INTO sessions (id, title, speaker, session_date)
VALUES (2, 'Building Reliable APIs', 'Jordan Patal', '2026-05-02'),
       (3, 'Cybersecurity Basics for Teams', 'Elena Rivera', '2026-05-03');

Running query in 'sqlite:///../data/conference.db'

2 rows affected.

++
||
++
++

## Tasks 5-6. Update a title + view it

Task: rename Jordan's talk, then show only his title.

New: `WHERE` also works with `SELECT`, to filter which rows come back (same as in `UPDATE` / `DELETE`).

In [12]:
%%sql
UPDATE sessions
SET title = 'Building Reliable REST APIs'
WHERE speaker = 'Jordan Patal';

SELECT title FROM sessions WHERE speaker = 'Jordan Patal';

Running query in 'sqlite:///../data/conference.db'

1 rows affected.

title
Building Reliable REST APIs


## Tasks 7-8. Add rooms

Task: add a `room` column, then assign Room A / B / C.

**Mistake: typo `'ELena Rivera'`.** Text comparison with `=` is case-sensitive, so this `UPDATE` matched **0 rows**: no error, nothing changed.
Elena's room stays `NULL` (check the output). It didn't matter here because her session is deleted in task 9,
but in real data this fails silently. Always check the result with a `SELECT` after an `UPDATE`.

Kept as I wrote it so the output shows the effect.

In [13]:
%%sql
ALTER TABLE sessions
ADD COLUMN room TEXT;

UPDATE sessions SET room = 'Room A' WHERE speaker = 'Maya Chen';
UPDATE sessions SET room = 'Room B' WHERE speaker = 'Jordan Patal';
UPDATE sessions SET room = 'Room C' WHERE speaker = 'ELena Rivera';

SELECT * FROM sessions;   -- Elena: room is NULL

Running query in 'sqlite:///../data/conference.db'

1 rows affected.

1 rows affected.

id,title,speaker,session_date,room
1,Intro to Data Visualization,Maya Chen,2026-05-01,Room A
2,Building Reliable REST APIs,Jordan Patal,2026-05-02,Room B
3,Cybersecurity Basics for Teams,Elena Rivera,2026-05-03,None


Safer alternative: filter on `id` instead of a typed name. IDs can't be misspelled the same way:

    UPDATE sessions SET room = 'Room C' WHERE id = 3;

## Tasks 9-10. Cancel a session + final schedule

Task: delete Elena's session, then view the final schedule.

In [14]:
%%sql
DELETE FROM sessions WHERE speaker = 'Elena Rivera';

SELECT * FROM sessions;   -- 2 rows left: Maya (Room A), Jordan (Room B)

Running query in 'sqlite:///../data/conference.db'

1 rows affected.

id,title,speaker,session_date,room
1,Intro to Data Visualization,Maya Chen,2026-05-01,Room A
2,Building Reliable REST APIs,Jordan Patal,2026-05-02,Room B
